In [9]:
%load_ext autoreload
%autoreload 2

In [10]:
import numpy as np
from typing import List, Tuple, Any, Optional, Dict, Union
import random
import math
import torch
import torch.nn as nn

In [11]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Utils

Refactor some utils like the spherical lloyd algorithm

In [6]:
# Original
import sys
sys.path.append('/home/arthur/Documents/Code/Github/KPConv-PyTorch/')


from kernels.kernel_points import (
    create_3D_rotations,
    spherical_Lloyd, 
    kernel_point_optimization_debug,
    load_kernels,
)

### Refactor initialization of kernel points

#### spherical Lloyd

In [150]:
# Original spherical Lloyd
set_seed(42)

radius = 1.0
num_points = 10
dimension = 3
fixed = 'center'
approximation = 'monte-carlo'
approx_n = 5000
max_iter = 500
momentum = 0.9
verbose = 0

# Radius used for optimization (points are rescaled afterwards)
radius0 = 1.0

# Random kernel points (Uniform distribution in a sphere)
kernel_points = np.zeros((0, dimension))
while kernel_points.shape[0] < num_points:
    new_points = np.random.rand(num_points, dimension) * 2 * radius0 - radius0
    kernel_points = np.vstack((kernel_points, new_points))
    d2 = np.sum(np.power(kernel_points, 2), axis=1)
    kernel_points = kernel_points[np.logical_and(d2 < radius0 ** 2, (0.9 * radius0) ** 2 < d2), :]
kernel_points = kernel_points[:num_points]

print(f"{kernel_points = }")

kernel_points = array([[ 0.19731697, -0.68796272, -0.68801096],
       [ 0.02846888,  0.18482914, -0.90709917],
       [-0.39077246, -0.80465577,  0.36846605],
       [ 0.24659625, -0.33820395, -0.8728833 ],
       [ 0.04546566, -0.14491796, -0.94916175],
       [ 0.26680751,  0.74292118,  0.60734415],
       [-0.77989615, -0.54412967, -0.14578442],
       [ 0.67060499, -0.35843987, -0.62696298],
       [ 0.75467871, -0.48411674,  0.31996809],
       [ 0.80083611,  0.26620291, -0.32194042]])


In [212]:

def random_spherical_points_numpy(
    num_points: int,
    radius: float = 1.0,
    ratio: Union[float, Tuple[float, float]] = 1.0,
    limits: Optional[Tuple[float, float]] = None,
) -> np.ndarray:
    if isinstance(ratio, float):
        ratio = (0.0, ratio)

    inner_limit, outer_limit = ratio
    
    points= np.zeros((0, 3))
    while points.shape[0] < num_points:
        # Generate random points in the bounding cube
        new_points = np.random.rand(num_points, 3) * 2 * radius - radius
        d2 = np.sum(np.power(new_points, 2), axis=1)

        # Filter points that fall within the spherical shell or full sphere as per the given range
        valid_points = new_points[np.logical_and(d2 < (outer_limit * radius) ** 2, d2 > (inner_limit * radius) ** 2)]
        points = np.vstack((points, valid_points))

    points = points[:num_points]
    
    return points


set_seed(42)
kernel_points_v2 = random_spherical_points_numpy(
    num_points=num_points, 
    radius=radius0, 
    ratio=(0.9, 1.0), 
)

print(f"{kernel_points_v2 = }")

kernel_points_v2 = array([[ 0.19731697, -0.68796272, -0.68801096],
       [ 0.02846888,  0.18482914, -0.90709917],
       [-0.39077246, -0.80465577,  0.36846605],
       [ 0.24659625, -0.33820395, -0.8728833 ],
       [ 0.04546566, -0.14491796, -0.94916175],
       [ 0.26680751,  0.74292118,  0.60734415],
       [-0.77989615, -0.54412967, -0.14578442],
       [ 0.67060499, -0.35843987, -0.62696298],
       [ 0.75467871, -0.48411674,  0.31996809],
       [ 0.80083611,  0.26620291, -0.32194042]])


In [152]:
np.allclose(kernel_points, kernel_points_v2)

True

#### For kernel points optimization debug

In [124]:
# Original
set_seed(42)

radius = 1.0
num_points = 10
dimension = 3
num_kernels = 1
fixed = 'center'
ratio = 0.66
verbose = 0

radius0 = 1.0
diameter0 = 2.0
moving_factor = 1e-2
continuous_moving_decay = 0.9995
thresh = 1e-5
clip = 0.05 * radius0

# Initialize random kernel points
kernel_points = np.random.rand(num_kernels * num_points - 1, dimension) * diameter0 - radius0
while kernel_points.shape[0] < num_kernels * num_points:
    new_points = np.random.rand(num_kernels * num_points - 1, dimension) * diameter0 - radius0
    kernel_points = np.vstack((kernel_points, new_points))
    d2 = np.sum(np.power(kernel_points, 2), axis=1)
    kernel_points = kernel_points[d2 < 0.5 * radius0 * radius0, :]
kernel_points = kernel_points[:num_kernels * num_points, :]

print(f"{kernel_points = }")

kernel_points = array([[-0.13610996, -0.41754172,  0.22370579],
       [ 0.32504457, -0.37657785,  0.04013604],
       [-0.28649335, -0.43813098,  0.08539217],
       [-0.37803536, -0.34963336,  0.45921236],
       [ 0.1225544 ,  0.54193436, -0.01240881],
       [ 0.02149461, -0.16517799, -0.55578438],
       [-0.35359414,  0.03758124,  0.40603792],
       [-0.49643541, -0.00550299, -0.39824338],
       [ 0.4564327 , -0.26443373,  0.26461166],
       [ 0.6344444 ,  0.11040162,  0.05930116]])


In [147]:
def random_spherical_points(
    num_points: int,
    radius: float,
    bounds: Union[float, Tuple[float, float]] = 1.0,
) -> np.ndarray:
    """Generate random points inside a sphere or a spherical shell based on radius limits.

    Args:
        num_points: The number of points to generate.
        radius: The radius of the sphere.
        bounds: A float or tuple of floats defining the inner and outer bounds of the sphere.
            If a single float is provided, it is treated as the outer limit, with the inner limit as 0.
            Defaults to 1.0.
    
    Returns:
        Generated points of shape (num_points, dimension).
    """

    if isinstance(bounds, float):
        bounds = (0.0, bounds)

    inner_bound, outer_bound = bounds
    
    points = torch.zeros((0, 3))  # Initialize an empty tensor for points
    while points.shape[0] < num_points:
        # Generate random points in the bounding cube
        new_points = (torch.rand(num_points, 3) * 2 * radius) - radius
        d2 = torch.sum(new_points**2, dim=1)

        # Filter points that fall within the spherical shell or full sphere as per the given range
        valid_points = new_points[(d2 < (outer_bound * radius) ** 2) & (d2 > (inner_bound * radius) ** 2)]
        points = torch.cat((points, valid_points), dim=0)

    points = points[:num_points]  # Ensure the exact number of points
    
    return points


set_seed(42)
kernel_points_v2 = random_spherical_points(
    num_points=num_points, 
    radius=radius0, 
    bounds=(0, math.sqrt(0.5)), 
)

print(f"{kernel_points_v2 = }")

kernel_points_v2 = tensor([[-0.4668,  0.2549, -0.4607],
        [-0.3154,  0.2687, -0.2712],
        [ 0.2880,  0.4142,  0.3163],
        [-0.3444,  0.3064, -0.2083],
        [ 0.2114, -0.2550,  0.5961],
        [ 0.0028, -0.3721, -0.0693],
        [ 0.2861, -0.2184,  0.3893],
        [-0.1727,  0.2089,  0.5163],
        [ 0.2517, -0.4301, -0.1096],
        [-0.0478,  0.5584, -0.2555]])


In [126]:
np.allclose(kernel_points, kernel_points_v2)

True

In [146]:
from torch_pointcloud.utils.plotly import plot_points


points = random_spherical_points(
    num_points=1000, 
    dimension=dimension, 
    radius=1, 
    bounds=(0.9, 1.0),
)


plot_points(points)

### Kernel point optimization

In [267]:
def kernel_point_optimization(
    radius: float,
    num_points: int,
    num_kernels: int = 1,
    dimension: int = 3,
    fixed: str = 'center',
    ratio: float = 0.66
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Creation of kernel points via optimization of potentials.

    Args:
        radius: Radius of the kernels.
        num_points: Number of points composing the kernels.
        num_kernels: Number of kernels to generate.
        dimension: Dimension of the space.
        fixed: Fix position of certain kernel points ('none', 'center', or 'verticals').
        ratio: Ratio of the radius where you want the kernel points to be placed.

    Returns:
        A tuple containing:
            - Optimized kernel points of shape [num_kernels, num_points, dimension].
            - Saved gradient norms of the optimization process.
    """
    
    # Optimization parameters
    radius0 = 1.0
    diameter0 = 2.0
    moving_factor = 1e-2
    continuous_moving_decay = 0.9995
    thresh = 1e-5
    clip = 0.05 * radius0

    # Initialize random kernel points
    kernel_points = np.random.rand(num_kernels * num_points - 1, dimension) * diameter0 - radius0
    while kernel_points.shape[0] < num_kernels * num_points:
        new_points = np.random.rand(num_kernels * num_points - 1, dimension) * diameter0 - radius0
        kernel_points = np.vstack((kernel_points, new_points))
        d2 = np.sum(np.power(kernel_points, 2), axis=1)
        kernel_points = kernel_points[d2 < 0.5 * radius0 * radius0, :]
    kernel_points = kernel_points[:num_kernels * num_points, :].reshape((num_kernels, num_points, -1))

    # Fix certain kernel points if specified
    if fixed == 'center':
        kernel_points[:, 0, :] *= 0
    if fixed == 'verticals':
        kernel_points[:, :3, :] *= 0
        kernel_points[:, 1, -1] += 2 * radius0 / 3
        kernel_points[:, 2, -1] -= 2 * radius0 / 3

    # Kernel optimization
    saved_gradient_norms = np.zeros((10000, num_kernels))
    old_gradient_norms = np.zeros((num_kernels, num_points))
    step = -1

    while step < 10000:
        step += 1

        # Compute gradients
        A = np.expand_dims(kernel_points, axis=2)
        B = np.expand_dims(kernel_points, axis=1)
        interd2 = np.sum(np.power(A - B, 2), axis=-1)
        inter_grads = (A - B) / (np.power(np.expand_dims(interd2, -1), 3 / 2) + 1e-6)
        inter_grads = np.sum(inter_grads, axis=1)

        circle_grads = 10 * kernel_points

        # All gradients
        gradients = inter_grads + circle_grads

        if fixed == 'verticals':
            gradients[:, 1:3, :-1] = 0

        # Compute norm of gradients
        gradients_norms = np.sqrt(np.sum(np.power(gradients, 2), axis=-1))
        saved_gradient_norms[step, :] = np.max(gradients_norms, axis=1)

        # Stop condition
        if fixed == 'center' and np.max(np.abs(old_gradient_norms[:, 1:] - gradients_norms[:, 1:])) < thresh:
            break
        elif fixed == 'verticals' and np.max(np.abs(old_gradient_norms[:, 3:] - gradients_norms[:, 3:])) < thresh:
            break
        elif np.max(np.abs(old_gradient_norms - gradients_norms)) < thresh:
            break

        old_gradient_norms = gradients_norms

        # Move points
        moving_dists = np.minimum(moving_factor * gradients_norms, clip)

        if fixed == 'center':
            moving_dists[:, 0] = 0
        if fixed == 'verticals':
            moving_dists[:, 0] = 0

        kernel_points -= np.expand_dims(moving_dists, -1) * gradients / np.expand_dims(gradients_norms + 1e-6, -1)

        # Moving factor decay
        moving_factor *= continuous_moving_decay

    # Remove unused lines in the saved gradients
    if step < 10000:
        saved_gradient_norms = saved_gradient_norms[:step + 1, :]

    # Rescale radius to fit the wanted ratio of radius
    r = np.sqrt(np.sum(np.power(kernel_points, 2), axis=-1))
    kernel_points *= ratio / np.mean(r[:, 1:])

    # Rescale kernels with real radius
    return kernel_points * radius, saved_gradient_norms


set_seed(42)
kernel_points, saved_gradient_norms = kernel_point_optimization(
    radius=1.0,
    num_points=10,
    num_kernels=1,
    dimension=3,
    fixed='center',
    ratio=0.66,
)

print(f"{kernel_points = }")

kernel_points = array([[[ 0.        ,  0.        ,  0.        ],
        [ 0.37299413, -0.54306198, -0.00129056],
        [-0.31443067, -0.50028142, -0.2990839 ],
        [-0.29606397, -0.39011744,  0.44081876],
        [ 0.06703513,  0.64405153, -0.12285586],
        [ 0.07856441,  0.03180909, -0.65334399],
        [-0.31033923,  0.36197091,  0.45925462],
        [-0.59284425,  0.18934779, -0.21644799],
        [ 0.36806846,  0.05628052,  0.54382688],
        [ 0.62699197,  0.14985694, -0.15099412]]])


In [252]:
plot_points(kernel_points[0])

In [275]:
# TODO: add type hints for return_grads (bool)) which returns the gradient norms if True
def gradient_optimization_spherical_points(
    radius: float,
    num_points: int,
    fixed: str = 'center',
    ratio: float = 0.66,
    max_steps: int = 10_000,
    step_size: float = 1e-2,
    step_decay: float = 0.9995,
    convergence_threshold: float = 1e-5,
    max_step_size: Optional[float] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """Creation of kernel points via optimization of potentials for a single kernel.

    Args:
        radius: Radius of the kernel.
        num_points: Number of points composing the kernel.
        fixed: Fix position of certain kernel points ('none', 'center', or 'verticals').
        ratio: Ratio of the radius where you want the kernel points to be placed.
        max_steps: Maximum number of optimization steps.
        step_size: Step size for moving points based on gradient norms.
        step_decay: Decay factor for reducing the step size over time.
        convergence_threshold: Threshold for stopping the optimization when gradient norm changes are small.
        max_step_size: Maximum distance a point can move in a single step.

    Returns:
        A tuple containing:
            - Optimized kernel points of shape [num_points, dimension].
            - Saved gradient norms of the optimization process.
    """
    def compute_gradients(points: np.ndarray) -> np.ndarray:
        A = np.expand_dims(points, axis=1)
        B = np.expand_dims(points, axis=0)
        interd2 = np.sum(np.power(A - B, 2), axis=-1)
        inter_grads = (A - B) / (np.power(np.expand_dims(interd2, -1), 3 / 2) + 1e-6)
        inter_grads = np.sum(inter_grads, axis=0)
        circle_grads = 10 * points

        return inter_grads + circle_grads
    
    # Parameters
    if max_step_size is None:
        max_step_size = 0.05 * radius

    # Initialize kernel points
    kernel_points = random_spherical_points_numpy(num_points, radius, ratio=(0, 0.7071067811865476))

    # Apply fixed positions if required
    if fixed == 'center':
        kernel_points[0, :] = 0  # Fix the first point to the center
    elif fixed == 'verticals':
        kernel_points[:3, :] = 0  # Fix the first three points
        kernel_points[1, -1] += 2 * radius / 3  # Move second point up vertically
        kernel_points[2, -1] -= 2 * radius / 3  # Move third point down vertically

    # Kernel optimization
    saved_grad_norms = np.zeros((max_steps,))
    old_grad_norms = np.zeros((num_points,))
    step = 0

    while step < max_steps:
        grads = compute_gradients(kernel_points)
        
        if fixed == 'verticals':
            grads[1:3, :-1] = 0
        
        grad_norms = np.sqrt(np.sum(np.power(grads, 2), axis=-1))
        saved_grad_norms[step] = np.max(grad_norms)

        # Check for stopping conditions
        if fixed == 'center' and np.max(np.abs(old_grad_norms[1:] - grad_norms[1:])) < convergence_threshold:
            break
        elif fixed == 'verticals' and np.max(np.abs(old_grad_norms[3:] - grad_norms[3:])) < convergence_threshold:
            break
        elif np.max(np.abs(old_grad_norms - grad_norms)) < convergence_threshold:
            break

        old_grad_norms = grad_norms

        # Move points
        moving_dists = np.minimum(step_size * grad_norms, max_step_size)
        if fixed == 'center' or fixed == 'verticals':
            moving_dists[0] = 0  # Do not move the first point if fixed

        kernel_points -= np.expand_dims(moving_dists, -1) * grads / np.expand_dims(grad_norms + 1e-6, -1)
        step_size *= step_decay
        step += 1

    # Rescale kernel points
    r = np.sqrt(np.sum(np.power(kernel_points, 2), axis=-1))
    kernel_points *= ratio / np.mean(r[1:])

    return kernel_points, saved_grad_norms[step - 1]


set_seed(42)
kernel_points_v2, grad = gradient_optimization_spherical_points(
    radius=1.0,
    num_points=10,
    fixed='center',
    ratio=0.66,
)

In [276]:
np.allclose(kernel_points, kernel_points_v2)

True

In [16]:
from torch_pointcloud.utils.geometry import gradient_optimization_spherical_points, spherical_lloyd
from torch_pointcloud.utils.plotly import plot_points

kernel_points_v2, grad = gradient_optimization_spherical_points(
    radius=1.0,
    num_points=100,
    fixed='none',
    ratio=0.66,
)

plot_points(kernel_points_v2)

In [20]:
from torch_pointcloud.utils.geometry import gradient_optimization_spherical_points, spherical_lloyd

kernel_points = spherical_lloyd(
    num_points=100,
    radius=1.0,
    position='center',
)

plot_points(kernel_points)


### Load Kernels

In [278]:
# Refactored
from pathlib import Path

from torch_pointcloud.utils.geometry import spherical_lloyd, rodrigues_rotation_matrices

CACHE_DIR = Path.home() / '.cache' / 'torch_pointcloud'


def kernel_points(
    radius: float,
    num_points: int,
    fixed: str,
    method: str = 'lloyd', # 'lloyd' or 'gradient'
):
    # Too many points switch to Lloyds
    if num_points > 30:
        method = 'lloyd'

    kernel_path = Path(CACHE_DIR, 'kernels', f"k_{num_points:03d}_{fixed}_{method}.pt")

    # Check if already done
    if kernel_path.exists():
        kernel_points = torch.load(kernel_path)
    else:
        if method == "lloyd":
            kernel_points = spherical_Lloyd(1.0,num_points, dimension=dimension,  fixed=fixed,)
        else:
            kernel_points, _ = kernel_point_optimization_debug(1.0,  num_points, fixed=fixed)

        kernel_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(kernel_points, kernel_path)

    # Random rotations for the kernel
    R = torch.eye(dimension)
    theta = random.random() * 2 * np.pi
    
    if fixed != 'vertical':
        c, s = torch.cos(theta), torch.sin(theta)
        R = torch.tensor([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=torch.float32)

    else:
        phi = (random.random() - 0.5) * np.pi
        # Create the first vector in carthesian coordinates
        u = torch.tensor([torch.cos(theta) * torch.cos(phi), torch.sin(theta) * torch.cos(phi), torch.sin(phi)])
        # Choose a random rotation angle
        alpha = random.random() * 2 * np.pi

        # Create the rotation matrix with this vector and angle
        # TODO: refactor this one using rotations
        R = create_3D_rotations(np.reshape(u, (1, -1)), np.reshape(alpha, (1, -1)))[0]

    # Add a small noise
    kernel_points += torch.normal(mean=0, std=0.01, size=kernel_points.shape)
    # Scale kernels
    kernel_points *= radius
    # Rotate kernels
    kernel_points = torch.matmul(kernel_points, R)

    return kernel_points